# DepthWizard -- Stage 1 Depth Model Fine-tuning

Fine-tunes the Depth Anything V2 Small backbone used by
`backend/app/pipeline/stage1_depth.py` (`load_depth_model` /
`run_inference`) -- this model IS the elevation extraction: RGB in,
relative height map out. Everything downstream (Stage 2 calibration,
Stage 3 shape cleanup, the whole visual output) is only as good as this
model's raw height accuracy.

**Read `AGENT_FINETUNE_GUIDE.md` in this same folder before you start** --
same run -> screenshot -> share-with-agent -> update loop as the
segmentation notebook, adapted for a regression model instead of
classification.

**History this notebook continues from** (from `stage1_depth.py`'s own
docstring -- read this before changing the loss or freeze depth):
- v1 -> v2: corrected a class-ID bug (v2 accidentally weighted "road"
  instead of "tree" in its hard-class loss term -- GAMUS's real mapping is
  `1=ground 2=low_vegetation 3=building 4=water 5=road 6=tree`, confirmed
  against the GAMUS paper, arXiv 2305.14914)
- v2 -> v3: building barely improved despite looking visually sharper
  (multi-scale gradient loss fixed edges, not absolute height) -> v3
  weights BOTH building AND tree in the hard-class loss term, not just one
- v3 -> v4 (**current production checkpoint**): unfroze more of the
  backbone (last 12 blocks instead of 6), same three-term loss
- v4 -> v5 (**abandoned**): added a `structure_loss` term to try to make
  the depth model itself output flat-topped buildings. This REGRESSED MAE
  across every land-cover class on re-evaluation. v4 was kept, and roof
  flattening is instead handled downstream in Stage 3
  (`stage3_mesh_prep.flatten_planar_classes`), which fixes shape without
  touching the depth model's numeric accuracy.

**Do not re-add a structure/edge-sharpening loss term here.** That
experiment already ran and regressed. If shape/roofline quality is the
goal, that's Stage 3's job, not this notebook's -- see
`AGENT_FINETUNE_GUIDE.md`'s note on this.

**Runtime:** Runtime -> Change runtime type -> GPU (T4 minimum; this
model + 512x512 tiles is heavier than the segmentation notebook -- an
A100/L4 will be meaningfully faster if available on your plan).

**Hugging Face token required** -- GAMUS downloads are much faster and
more reliable with one. Click the key icon in Colab's left sidebar ->
"+ New secret" -> name it exactly `HF_TOKEN` -> paste a token from
https://huggingface.co/settings/tokens -> enable notebook access.


## 1. Setup

In [ ]:
!pip install -q h5py huggingface_hub transformers torch

import os
import json
import time
import random

import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from huggingface_hub import HfApi, hf_hub_download
from transformers import AutoModelForDepthEstimation

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device != "cuda":
    print("WARNING: no GPU detected -- Runtime > Change runtime type > GPU")

# Confirmed class ID mapping (GAMUS paper, arXiv 2305.14914 -- matches
# backend/app/config.py's LANDCOVER_CLASS_IDS exactly, keep these in sync)
UNLABELED_ID, GROUND_ID, LOW_VEG_ID, BUILDING_ID, WATER_ID, ROAD_ID, TREE_ID = 0, 1, 2, 3, 4, 5, 6
CLASS_NAMES = {GROUND_ID: "ground", LOW_VEG_ID: "low_vegetation", BUILDING_ID: "building",
               WATER_ID: "water", ROAD_ID: "road", TREE_ID: "tree"}


## 2. Config

Edit these. `RUN_NAME` matters most if you're one of several people
fine-tuning in parallel -- give each approach a distinct name so
checkpoints/exports never collide.


In [ ]:
# ---- identity for this run ----
RUN_NAME = "v5_attempt_a"       # e.g. "v5_wider_unfreeze", "v5_higher_hard_weight" -- change per attempt

# ---- starting point: continue from the current production checkpoint ----
# Upload depth_anything_v2_gamus_v4.pth via the file browser on the left
# (or mount Drive below), then set this path. This is the RECOMMENDED
# default -- training from scratch discards 4 rounds of validated
# improvement for no reason unless you specifically want to try a
# different base recipe.
RESUME_FROM_CHECKPOINT = "/content/depth_anything_v2_gamus_v4.pth"

# ---- data ----
REPO_ID = "earthflow/GAMUS"
SPLIT = "train"
CANDIDATE_POOL_SIZE = 500
N_TILES = 350
MIN_HARD_CLASS_TILES = 150   # guarantee this many tiles with strong building+tree presence
IMG_SIZE = 512               # must match stage1_depth.MODEL_INPUT_SIZE -- do not change
BATCH_SIZE = 4

# ---- model ----
MODEL_NAME = "depth-anything/Depth-Anything-V2-Small-hf"
# v4 (current) unfroze the last 12 backbone blocks + head/neck. This is
# the proven starting point -- change it deliberately, one run at a time,
# not as a first guess.
LAST_N_BLOCKS_TO_UNFREEZE = 12

# ---- loss -- same three-term loss as v3/v4, see the loss section below
# for what each term does. HARD_CLASS_WEIGHT applies to BOTH building AND
# tree pixels (the "tall, long-tail" classes) -- this is what v3 fixed
# vs v2's single-class weighting.
HARD_CLASS_WEIGHT = 3.0
GRAD_LOSS_WEIGHT = 0.5
HARD_PENALTY_WEIGHT_COEF = 1.0

# ---- training ----
N_EPOCHS = 30
LEARNING_RATE = 1e-4
PATIENCE = 5   # early stop after this many epochs with no val improvement

# ---- output ----
OUTPUT_DIR = f"/content/checkpoints_{RUN_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
CHECKPOINT_NAME = f"depth_anything_v2_gamus_{RUN_NAME}.pth"   # distinct per run -- never overwrite v4 directly
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, CHECKPOINT_NAME)

random.seed(0); np.random.seed(0); torch.manual_seed(0)
print(f"Run: {RUN_NAME}  ->  will save to {CHECKPOINT_PATH}")


## 3. Dataset -- GAMUS subset, biased toward building + tree coverage

Building and tree are the two "hard" classes here: both are tall and
long-tail-distributed in height (most of a scene is near-zero-height
ground/road/low-vegetation), so a naive random sample under-represents
exactly the pixels this model struggles most with. This scans a candidate
pool and guarantees a meaningful number of tiles with high combined
building+tree coverage, rather than alphabetical/random tile order.

This downloads real triples (RGB + nDSM height + class mask) via
`hf_hub_download`, which caches to local disk -- unlike the segmentation
notebook's custom cache, this reuses `hf_hub_download`'s own cache, so
re-running this cell in the same session is fast (no re-download) and a
fresh Colab session re-downloads once.


In [ ]:
api = HfApi()
all_files = api.list_repo_files(REPO_ID, repo_type="dataset")
rgb_files = sorted([f for f in all_files if f.startswith(f"images/{SPLIT}/") and f.endswith(".h5")])
candidate_stems = [f.replace(f"images/{SPLIT}/", "").replace("_RGB.h5", "") for f in rgb_files[:CANDIDATE_POOL_SIZE]]

def find_matching_file(stem, folder):
    matches = [f for f in all_files if f.startswith(f"{folder}/{SPLIT}/{stem}")]
    if len(matches) != 1:
        raise ValueError(f"Expected 1 match for {stem} in {folder}, got {matches}")
    return matches[0]

print(f"Scanning {len(candidate_stems)} candidate tiles for building+tree coverage...")
hard_class_fraction = {}
for stem in candidate_stems:
    class_path = hf_hub_download(REPO_ID, find_matching_file(stem, "classes"), repo_type="dataset")
    with h5py.File(class_path, "r") as f:
        mask = np.array(f["image"])
    hard_class_fraction[stem] = float(np.mean((mask == BUILDING_ID) | (mask == TREE_ID)))

sorted_by_hard = sorted(hard_class_fraction.items(), key=lambda x: -x[1])
hard_heavy_stems = [s for s, frac in sorted_by_hard[:MIN_HARD_CLASS_TILES]]
remaining_stems = [s for s in candidate_stems if s not in hard_heavy_stems]
final_stems = hard_heavy_stems + remaining_stems[: N_TILES - len(hard_heavy_stems)]

print(f"Final subset: {len(final_stems)} tiles, {len(hard_heavy_stems)} building+tree-heavy "
      f"(top combined fraction: {sorted_by_hard[0][1]:.2f}, "
      f"median: {np.median(list(hard_class_fraction.values())):.3f})")


In [ ]:
local_paths = []
for stem in final_stems:
    rgb_path = hf_hub_download(REPO_ID, find_matching_file(stem, "images"), repo_type="dataset")
    height_path = hf_hub_download(REPO_ID, find_matching_file(stem, "heights"), repo_type="dataset")
    class_path = hf_hub_download(REPO_ID, find_matching_file(stem, "classes"), repo_type="dataset")
    local_paths.append((rgb_path, height_path, class_path))

total_mb = sum(os.path.getsize(p) for triple in local_paths for p in triple) / 1e6
print(f"Downloaded {len(local_paths)} triples, {total_mb:.1f} MB total")


In [ ]:
class GamusDataset(Dataset):
    def __init__(self, triples, img_size=IMG_SIZE):
        self.triples = triples
        self.img_size = img_size

    def __len__(self):
        return len(self.triples)

    def __getitem__(self, idx):
        rgb_path, height_path, class_path = self.triples[idx]
        with h5py.File(rgb_path, "r") as f:
            rgb = np.array(f["image"]).astype(np.float32) / 255.0
        with h5py.File(height_path, "r") as f:
            ndsm = np.array(f["image"]).astype(np.float32)
        with h5py.File(class_path, "r") as f:
            landcover = np.array(f["image"]).astype(np.int64)

        rgb_t = torch.from_numpy(rgb).permute(2, 0, 1)
        ndsm_t = torch.from_numpy(ndsm).unsqueeze(0)
        landcover_t = torch.from_numpy(landcover).unsqueeze(0)

        rgb_t = torch.nn.functional.interpolate(rgb_t.unsqueeze(0), size=(self.img_size, self.img_size), mode="bilinear", align_corners=False).squeeze(0)
        ndsm_t = torch.nn.functional.interpolate(ndsm_t.unsqueeze(0), size=(self.img_size, self.img_size), mode="bilinear", align_corners=False).squeeze(0)
        landcover_t = torch.nn.functional.interpolate(landcover_t.unsqueeze(0).float(), size=(self.img_size, self.img_size), mode="nearest").squeeze(0).long()

        return rgb_t, ndsm_t, landcover_t

full_dataset = GamusDataset(local_paths)
val_size = max(1, int(0.2 * len(full_dataset)))
train_size = len(full_dataset) - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f"Train: {len(train_ds)} tiles ({len(train_loader)} batches/epoch), Val: {len(val_ds)} tiles")


## 4. Model + freeze strategy

Loads the base Depth Anything V2 Small architecture, then applies
`RESUME_FROM_CHECKPOINT`'s weights on top (if given) -- exactly the same
load path `stage1_depth.load_depth_model()` uses in production, so what
trains here is a direct continuation of the live checkpoint, not a
diverged copy.


In [ ]:
model = AutoModelForDepthEstimation.from_pretrained(MODEL_NAME)

if RESUME_FROM_CHECKPOINT is not None:
    state_dict = torch.load(RESUME_FROM_CHECKPOINT, map_location=device)
    model.load_state_dict(state_dict)
    print(f"Resumed from {RESUME_FROM_CHECKPOINT}")
else:
    print("Training from the zero-shot base model (no RESUME_FROM_CHECKPOINT set)")

model.to(device)

for param in model.parameters():
    param.requires_grad = False
for name, param in model.named_parameters():
    if "head" in name or "neck" in name:
        param.requires_grad = True

backbone_layer_ids = sorted(set(
    name.split(".layer.")[1].split(".")[0]
    for name, _ in model.named_parameters()
    if "backbone" in name and ".layer." in name
), key=int)
unfreeze_ids = set(backbone_layer_ids[-LAST_N_BLOCKS_TO_UNFREEZE:]) if backbone_layer_ids else set()
for name, param in model.named_parameters():
    if "backbone" in name and ".layer." in name:
        if name.split(".layer.")[1].split(".")[0] in unfreeze_ids:
            param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)  "
      f"(last {LAST_N_BLOCKS_TO_UNFREEZE} backbone blocks + head/neck)")


## 5. Loss -- scale-invariant + multi-scale gradient + dual hard-class weighting

Three terms, unchanged from v3/v4 (see the history note at the top for
why v5's fourth term, structure_loss, is NOT here):

- **scale_invariant_loss**: the core depth-regression loss -- penalizes
  log-space error while being invariant to a global scale offset (since
  Stage 1 output is relative height anyway, absolute scale isn't this
  model's job -- Stage 2 calibration handles that).
- **multi_scale_gradient_loss**: penalizes gradient (edge) mismatches at
  multiple downsampled scales -- this is what actually sharpens
  boundaries/edges, complementary to the pointwise scale-invariant term.
- **hard_class_penalty**: extra weighted L1 penalty specifically on
  building + tree pixels (`HARD_CLASS_WEIGHT`) -- these are tall and
  underrepresented by pixel count, so without this term the loss is
  dominated by trivially-easy near-zero-height ground/road/low-vegetation
  pixels and building/tree accuracy stalls (this was v2's confirmed bug,
  see history note above).


In [ ]:
def scale_invariant_loss(pred, target, lam=0.5, eps=1e-6):
    pred = torch.clamp(pred, min=eps)
    target = torch.clamp(target, min=eps)
    d = torch.log(pred) - torch.log(target)
    n = d.numel()
    return torch.mean(d ** 2) - lam * (torch.sum(d) ** 2) / (n ** 2)


def multi_scale_gradient_loss(pred, target, n_scales=4, eps=1e-6):
    pred = torch.clamp(pred, min=eps)
    target = torch.clamp(target, min=eps)
    log_pred = torch.log(pred)
    log_target = torch.log(target)
    total = 0.0
    p, t = log_pred, log_target
    for scale in range(n_scales):
        p_dx = p[:, :, :, 1:] - p[:, :, :, :-1]
        p_dy = p[:, :, 1:, :] - p[:, :, :-1, :]
        t_dx = t[:, :, :, 1:] - t[:, :, :, :-1]
        t_dy = t[:, :, 1:, :] - t[:, :, :-1, :]
        total = total + torch.mean(torch.abs(p_dx - t_dx)) + torch.mean(torch.abs(p_dy - t_dy))
        if scale < n_scales - 1:
            p = torch.nn.functional.avg_pool2d(p, 2)
            t = torch.nn.functional.avg_pool2d(t, 2)
    return total / n_scales


def hard_class_penalty(pred, target, landcover_mask, weight=HARD_CLASS_WEIGHT, eps=1e-6):
    """Weighted L1 penalty on BUILDING + TREE pixels together -- the two tall,
    long-tail-distributed, historically-underperforming classes (see history note)."""
    pred = torch.clamp(pred, min=eps)
    target = torch.clamp(target, min=eps)
    abs_err = torch.abs(torch.log(pred) - torch.log(target))
    hard_mask = ((landcover_mask == BUILDING_ID) | (landcover_mask == TREE_ID)).float()
    if hard_mask.sum() == 0:
        return torch.tensor(0.0, device=pred.device)
    weighted = abs_err * hard_mask * weight
    return weighted.sum() / (hard_mask.sum() * weight + eps)


def combined_loss(pred, target, landcover_mask, grad_weight=GRAD_LOSS_WEIGHT,
                   hard_weight_coef=HARD_PENALTY_WEIGHT_COEF):
    si = scale_invariant_loss(pred, target)
    grad = multi_scale_gradient_loss(pred, target)
    hard_penalty = hard_class_penalty(pred, target, landcover_mask)
    return si + grad_weight * grad + hard_weight_coef * hard_penalty


## 6. Train loop -- early stopping, checkpoint saved as a raw state_dict

**Checkpoint format matters here**: saved as `model.state_dict()`
directly (`torch.save(model.state_dict(), ...)`), NOT wrapped in a dict
like the segmentation checkpoint. `stage1_depth.load_depth_model()` calls
`model.load_state_dict(torch.load(checkpoint_path))` directly on the
loaded object -- a wrapped dict would break that load. Don't change this
without also updating `load_depth_model()` to match.


In [ ]:
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, n_batches = 0.0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for rgb_batch, ndsm_batch, landcover_batch in loader:
            rgb_batch = rgb_batch.to(device)
            ndsm_batch = ndsm_batch.to(device)
            landcover_batch = landcover_batch.to(device)
            ndsm_shifted = ndsm_batch - ndsm_batch.min() + 1.0

            outputs = model(pixel_values=rgb_batch)
            pred = outputs.predicted_depth.unsqueeze(1)
            pred_resized = torch.nn.functional.interpolate(
                pred, size=ndsm_shifted.shape[-2:], mode="bicubic", align_corners=False
            )
            loss = combined_loss(pred_resized, ndsm_shifted, landcover_batch)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            n_batches += 1
    return total_loss / max(n_batches, 1)

train_losses, val_losses = [], []
best_val_loss = float("inf")
epochs_without_improvement = 0

for epoch in range(N_EPOCHS):
    t0 = time.time()
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    gap = train_loss - val_loss
    print(f"epoch {epoch+1:02d}/{N_EPOCHS}  train {train_loss:.4f}  val {val_loss:.4f}  "
          f"(train-val gap {gap:+.4f})  ({time.time()-t0:.0f}s)")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print("  -> saved checkpoint (best val so far)")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"  -> no val improvement for {PATIENCE} epochs, stopping early at epoch {epoch+1}")
            break

plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("epoch")
plt.ylabel("combined loss")
plt.legend()
plt.title(f"Run: {RUN_NAME}")
plt.savefig(os.path.join(OUTPUT_DIR, f"curves_{RUN_NAME}.png"), dpi=100, bbox_inches="tight")
plt.show()
print(f"\nTraining done. Best val loss: {best_val_loss:.4f}")


## 7. Per-class MAE evaluation -- SCREENSHOT THIS for your agent

Linear-scale-aligned MAE (median-ratio alignment, matching the
methodology `stage2_calibration`/the project's own evaluation reports
use) against the zero-shot base model AND the checkpoint you resumed
from, so you can see the actual delta this run produced, not just an
absolute number in isolation.


In [ ]:
N_EVAL_TILES = 30

def align_scale_linear(pred, target, eps=1e-6):
    pred_median = np.median(np.clip(pred, eps, None))
    target_median = np.median(np.clip(target, 0, None) + eps)
    scale = target_median / pred_median if pred_median > eps else 1.0
    return pred * scale

def masked_mae(pred, target, mask=None):
    if mask is not None:
        if mask.sum() == 0:
            return np.nan
        return np.mean(np.abs(pred[mask] - target[mask]))
    return np.mean(np.abs(pred - target))

def evaluate_model(eval_model, tag):
    eval_model.eval()
    overall_errs = []
    per_class_errs = {c: [] for c in CLASS_NAMES}
    eval_indices = val_ds.indices[:N_EVAL_TILES]
    for idx in eval_indices:
        rgb, ndsm, landcover = full_dataset[idx]
        ndsm_np = ndsm[0].numpy()
        landcover_np = landcover[0].numpy()
        with torch.no_grad():
            raw_pred = eval_model(pixel_values=rgb.unsqueeze(0).to(device)).predicted_depth
        pred_resized = torch.nn.functional.interpolate(
            raw_pred.unsqueeze(1), size=ndsm_np.shape, mode="bicubic", align_corners=False
        )[0, 0].cpu().numpy()
        pred_aligned = align_scale_linear(pred_resized, ndsm_np)

        overall_errs.append(masked_mae(pred_aligned, ndsm_np))
        for c in CLASS_NAMES:
            m = masked_mae(pred_aligned, ndsm_np, landcover_np == c)
            if not np.isnan(m):
                per_class_errs[c].append(m)

    row = f"{tag:22s}  overall {np.mean(overall_errs):6.3f}m"
    for c, name in CLASS_NAMES.items():
        vals = per_class_errs[c]
        row += f"   {name} {np.mean(vals):6.3f}m" if vals else f"   {name}   n/a"
    print(row)
    return np.mean(overall_errs)

print(f"Evaluating on {N_EVAL_TILES} validation tiles (linear-aligned MAE, lower is better)\n")

zero_shot_model = AutoModelForDepthEstimation.from_pretrained(MODEL_NAME).to(device)
evaluate_model(zero_shot_model, "Zero-shot")

if RESUME_FROM_CHECKPOINT is not None:
    baseline_model = AutoModelForDepthEstimation.from_pretrained(MODEL_NAME).to(device)
    baseline_model.load_state_dict(torch.load(RESUME_FROM_CHECKPOINT, map_location=device))
    evaluate_model(baseline_model, "Baseline (resumed-from)")

model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
this_run_mae = evaluate_model(model, f"This run ({RUN_NAME})")


## 8. Visual comparison -- SCREENSHOT THIS for your agent

RGB | zero-shot | this run | ground truth nDSM, on the validation tile with the highest building+tree coverage -- picked deliberately so the comparison is on the hardest, most informative tile rather than an easy near-flat one.

In [ ]:
hard_scores = []
for i in val_ds.indices:
    _, _, lc = full_dataset[i]
    hard_scores.append(float(((lc[0] == BUILDING_ID) | (lc[0] == TREE_ID)).float().mean()))
best_idx_in_val = int(np.argmax(hard_scores))
rgb_sample, ndsm_sample, landcover_sample = val_ds[best_idx_in_val]
print(f"Selected val tile with building+tree coverage: {hard_scores[best_idx_in_val]:.2f}")

rgb_input = rgb_sample.unsqueeze(0).to(device)
with torch.no_grad():
    zero_shot_pred = zero_shot_model(pixel_values=rgb_input).predicted_depth
    this_run_pred = model(pixel_values=rgb_input).predicted_depth

zero_shot_pred = torch.nn.functional.interpolate(zero_shot_pred.unsqueeze(1), size=ndsm_sample.shape[-2:], mode="bicubic", align_corners=False)[0, 0].cpu().numpy()
this_run_pred = torch.nn.functional.interpolate(this_run_pred.unsqueeze(1), size=ndsm_sample.shape[-2:], mode="bicubic", align_corners=False)[0, 0].cpu().numpy()

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(rgb_sample.permute(1, 2, 0).numpy()); axes[0].set_title("RGB input"); axes[0].axis("off")
axes[1].imshow(zero_shot_pred, cmap="inferno"); axes[1].set_title("Zero-shot"); axes[1].axis("off")
axes[2].imshow(this_run_pred, cmap="inferno"); axes[2].set_title(f"This run ({RUN_NAME})"); axes[2].axis("off")
im = axes[3].imshow(ndsm_sample[0].numpy(), cmap="terrain"); axes[3].set_title("Ground truth nDSM"); axes[3].axis("off")
plt.colorbar(im, ax=axes[3], shrink=0.7, label="meters")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f"visual_check_{RUN_NAME}.png"), dpi=100, bbox_inches="tight")
plt.show()
vis_path = os.path.join(OUTPUT_DIR, f"visual_check_{RUN_NAME}.png")
print(f"Saved to {vis_path} -- screenshot this cell output to share.")


## 9. Export

In [ ]:
# Option A: mount Google Drive and copy the checkpoint + screenshots there
from google.colab import drive
drive.mount("/content/drive")

import shutil
export_dir = f"/content/drive/MyDrive/depthwizard_checkpoints/{RUN_NAME}"
os.makedirs(export_dir, exist_ok=True)
for fname in os.listdir(OUTPUT_DIR):
    shutil.copy(os.path.join(OUTPUT_DIR, fname), os.path.join(export_dir, fname))
print(f"Exported to Google Drive: {export_dir}")


In [ ]:
# Option B: direct download to your machine instead of Drive
from google.colab import files
files.download(CHECKPOINT_PATH)


## 10. What to try next

Things worth experimenting with across runs -- change ONE at a time:

- `LAST_N_BLOCKS_TO_UNFREEZE` -- v3 used 6, v4 (current) used 12. More
  unfrozen blocks means more capacity to adapt but more risk of
  overfitting/forgetting general depth priors on this small a dataset --
  worth trying both directions (8 and 16) from v4's 12.
- `HARD_CLASS_WEIGHT` -- v3/v4 used 3.0 for both building+tree combined.
  If per-class eval (Section 7) shows building and tree have very
  different MAE from each other, consider SEPARATE weights per class
  instead of one shared weight (would need `hard_class_penalty` split
  into two masks) -- a real, scoped next experiment, not implemented here
  yet.
- `GRAD_LOSS_WEIGHT` -- controls how much edge-sharpness is worth trading
  against pointwise accuracy. Higher = sharper edges, possible pointwise
  regression; lower = smoother, possibly blurrier boundaries.
- `N_TILES` / `MIN_HARD_CLASS_TILES` -- more data, or an even stronger
  building+tree bias, if Colab session time allows.
- Do NOT add a 4th loss term aimed at building/roof SHAPE (flatness,
  edge-crispness beyond what multi_scale_gradient_loss already does) --
  that's the v5 experiment that already regressed MAE. Shape belongs in
  Stage 3 (`stage3_mesh_prep.flatten_planar_classes`), not here.

Every run needs its own `RUN_NAME` so checkpoints never overwrite each
other. Do NOT rename any output file to
`depth_anything_v2_gamus_v4.pth` yourselves -- report back the per-class
MAE table + visual grid first; that swap happens deliberately.
